# CallSense: likely escalation baselines

Binary target: **likely escalation** (`escalation=1`), including explicit escalation (manager requests, regulator complaints, stated escalation) and strong unresolved-service indicators (repeated failed contacts, prolonged unresolved failures).

This is a screening label, not proof of actual escalation or an instruction to escalate. Dissatisfaction alone does not automatically imply likely escalation. Existing labels are preserved; heuristic review flags are not relabeling decisions.

Training: `data/train_reviewed.csv`. Development: `data/processed/val.csv`. No team-call data or intent-model artifacts are used. Development results are not independent test results: this split has already informed earlier project work.

Run sequentially using **Python (CallSense)**. Dependencies: pandas, numpy, scikit-learn, joblib, nbformat/Jupyter. No Transformer or GPU is needed. Outputs are created only in the final export cell.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
import hashlib
import json
import re
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, f1_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks/07_escalation_baselines.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from inside the CallSense project")
TRAIN_PATH = ROOT / "data/train_reviewed.csv"
DEV_PATH = ROOT / "data/processed/val.csv"
SEED = 42
N_SPLITS = 3
C_VALUES = [0.1, 1.0, 10.0]
THRESHOLDS = [round(x / 100, 2) for x in range(10, 91, 5)]


## Validate inputs and separation

Use canonical author handles extracted from source URLs, rather than assigning unknown authors to independent groups. Stop if an author cannot be resolved. Check IDs, author separation, post URLs, and normalized duplicate texts across the existing splits. Validation does not move records or change labels.


In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def clean_text(text):
    # Same normalization as the existing keyword/TF-IDF baseline.
    text = text.lower()
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub("[أإآٱ]", "ا", text)
    return text.replace("ة", "ه").replace("ى", "ي").replace("ـ", "")

def author_from_url(value):
    url = urlparse(value if "://" in value else "https://" + value)
    parts = url.path.strip("/").split("/")
    host = url.netloc.lower().removeprefix("www.")
    if host in {"x.com", "twitter.com", "mobile.twitter.com", "mobile.x.com"} and len(parts) >= 3 and parts[1] == "status":
        return parts[0].lower()
    raise ValueError(f"Cannot resolve author from source URL: {value}")

def canonical_post(value):
    url = urlparse(value if "://" in value else "https://" + value)
    match = re.search(r"/status/(\d+)", url.path)
    return match.group(1) if match else url.netloc.lower() + url.path.rstrip("/")

input_hashes = {str(p.relative_to(ROOT)): sha256(p) for p in [TRAIN_PATH, DEV_PATH]}
train = pd.read_csv(TRAIN_PATH)
dev = pd.read_csv(DEV_PATH)
for name, frame in [("training", train), ("development", dev)]:
    required = {"complaint_id", "model_text", "source_url", "escalation"}
    if frame.empty or not required <= set(frame.columns):
        raise ValueError(f"{name}: empty input or missing required columns")
    for col in ["complaint_id", "model_text", "source_url"]:
        if frame[col].isna().any() or not frame[col].map(lambda x: isinstance(x, str) and bool(x.strip())).all():
            raise ValueError(f"{name}: missing or invalid {col}")
    if frame["complaint_id"].duplicated().any():
        raise ValueError(f"{name}: duplicate complaint IDs")
    if not frame["escalation"].isin([0, 1]).all():
        raise ValueError(f"{name}: escalation must be binary and nonmissing")
    frame["author_group"] = frame["source_url"].map(author_from_url)
    frame["canonical_post"] = frame["source_url"].map(canonical_post)
    frame["normalized_text"] = frame["model_text"].map(clean_text).str.replace(r"\s+", " ", regex=True).str.strip()
    if frame["normalized_text"].eq("").any():
        raise ValueError(f"{name}: empty normalized text")
for col in ["complaint_id", "author_group", "canonical_post", "normalized_text"]:
    if set(train[col]) & set(dev[col]):
        raise ValueError(f"Training/development overlap in {col}")
print("Input validation passed; no split or label changes")
display(pd.DataFrame({"training": train.escalation.value_counts(),
                      "development": dev.escalation.value_counts()}).fillna(0).astype(int))


## Existing keyword baseline and training-only ambiguity flags

The keyword rule is unchanged: `مشرف`, `مدير`, `شكوي رسميه`, `برفع شكوي`, using normalized substring matching. A rule match is a binary score, not a calibrated probability.

The broader review cues below only flag training examples for human inspection. Positive labels without an explicit cue can legitimately reflect unresolved service. Negative labels with such cues may be quoted, negated, or historical. Do not infer annotation errors from these rules. Development errors are inspected only after final model selection.


In [ ]:
KEYWORDS = ["مشرف", "مدير", "شكوي رسميه", "برفع شكوي"]

def keyword_predict(texts):
    return np.array([int(any(clean_text(word) in clean_text(text) for word in KEYWORDS))
                     for text in texts])

explicit_cues = ["تصعيد", "تم التصعيد", "مدير", "مشرف", "هيئة الاتصالات", "cst", "برفع شكوي"]
review_flags = train[["complaint_id", "model_text", "escalation"]].copy()
review_flags["explicit_cue"] = train.model_text.map(
    lambda t: any(clean_text(cue) in clean_text(t) for cue in explicit_cues))
review_flags["review_reason"] = np.where(
    (review_flags.escalation == 1) & ~review_flags.explicit_cue,
    "Positive without explicit cue: review strength of unresolved-service evidence",
    np.where((review_flags.escalation == 0) & review_flags.explicit_cue,
             "Negative with cue: review context, negation, and label definition", ""))
review_flags = review_flags.loc[review_flags.review_reason.ne("")].copy()
display(review_flags)


## Grouped cross-validation within training

Three folds stratify on escalation and group by connected author/post/normalized-text clusters. Every fold must contain both classes on both sides. Only training data participates in selection. Each candidate is cloned and fitted on each fold, so TF-IDF vocabulary and IDF are learned separately within that fold.

Select logistic-regression `C` and threshold by pooled out-of-fold **positive-class F1**. Fixed tie-break: smallest `C`, then threshold closest to 0.50, then lower threshold. The selected OOF score is a tuning score, not an unbiased generalization estimate. Record fold variation and counts; 20 positive training records still imply substantial uncertainty.


In [ ]:
y_train = train.escalation.to_numpy(dtype=int)
X_train = train.model_text.map(clean_text)
# Connected groups keep shared authors, posts, and normalized duplicates together.
parent = list(range(len(train)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for column in ["author_group", "canonical_post", "normalized_text"]:
    first_seen = {}
    for i, value in enumerate(train[column]):
        if value in first_seen:
            parent[find(i)] = find(first_seen[value])
        else:
            first_seen[value] = i
groups = np.array([find(i) for i in range(len(train))])
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = list(cv.split(X_train, y_train, groups))
fold_summary = []
for fold, (fit_idx, hold_idx) in enumerate(splits, 1):
    if set(groups[fit_idx]) & set(groups[hold_idx]):
        raise ValueError("Author overlap inside CV")
    if set(y_train[fit_idx]) != {0, 1} or set(y_train[hold_idx]) != {0, 1}:
        raise ValueError("Each CV partition must contain both escalation classes")
    fold_summary.append({"fold": fold, "fit_records": len(fit_idx),
                         "fit_positives": int(y_train[fit_idx].sum()),
                         "heldout_records": len(hold_idx),
                         "heldout_positives": int(y_train[hold_idx].sum())})
display(pd.DataFrame(fold_summary))

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000,
                                       random_state=SEED)),
])


In [ ]:
# Model fitting starts here. Do not execute during lightweight validation.
search_rows = []
oof_by_c = {}
for c in C_VALUES:
    oof = np.full(len(train), np.nan)
    seen = np.zeros(len(train), dtype=int)
    for fit_idx, hold_idx in splits:
        candidate = clone(baseline).set_params(classifier__C=c)
        candidate.fit(X_train.iloc[fit_idx], y_train[fit_idx])
        positive_index = list(candidate.classes_).index(1)
        oof[hold_idx] = candidate.predict_proba(X_train.iloc[hold_idx])[:, positive_index]
        seen[hold_idx] += 1
    if not np.isfinite(oof).all() or not (seen == 1).all():
        raise RuntimeError("Invalid out-of-fold coverage or probabilities")
    oof_by_c[c] = oof
    for threshold in THRESHOLDS:
        predicted = (oof >= threshold).astype(int)
        fold_f1 = [f1_score(y_train[idx], predicted[idx], zero_division=0) for _, idx in splits]
        search_rows.append({"C": c, "threshold": threshold,
                            "positive_f1": float(f1_score(y_train, predicted, zero_division=0)),
                            "precision": float(precision_score(y_train, predicted, zero_division=0)),
                            "recall": float(recall_score(y_train, predicted, zero_division=0)),
                            "fold_f1": fold_f1})
selected = sorted(search_rows, key=lambda r: (-r["positive_f1"], r["C"],
                                             abs(r["threshold"] - 0.5), r["threshold"]))[0]
selected_c = selected["C"]
selected_threshold = selected["threshold"]
print("Training-only selection:", selected)
selected_model = clone(baseline).set_params(classifier__C=selected_c)
selected_model.fit(X_train, y_train)


## One development evaluation after selection

Evaluate the fixed keyword rule and selected TF-IDF model once. Report positive-class precision/recall/F1, binary macro-F1, and a confusion matrix ordered `[0, 1]` (rows actual, columns predicted). **PR-AUC here means average precision**, not trapezoidal integration. Keyword average precision uses its binary rule score and has only two score levels.

Development has only five positives in the current snapshot, so one case changes recall by 20 percentage points. Do not select a different threshold or model after inspecting development results.


In [ ]:
def evaluate(y_true, scores, threshold):
    predictions = (scores >= threshold).astype(int)
    return predictions, {
        "positive_precision": float(precision_score(y_true, predictions, zero_division=0)),
        "positive_recall": float(recall_score(y_true, predictions, zero_division=0)),
        "positive_f1": float(f1_score(y_true, predictions, zero_division=0)),
        "macro_f1": float(f1_score(y_true, predictions, average="macro", zero_division=0)),
        "pr_auc_average_precision": float(average_precision_score(y_true, scores)),
        "confusion_matrix_labels": [0, 1],
        "confusion_matrix": confusion_matrix(y_true, predictions, labels=[0, 1]).tolist(),
        "positive_support": int(y_true.sum()), "records": len(y_true),
        "classification_report": classification_report(y_true, predictions, labels=[0, 1],
            target_names=["no_likely_escalation", "likely_escalation"],
            output_dict=True, zero_division=0),
    }

y_dev = dev.escalation.to_numpy(dtype=int)
keyword_scores = keyword_predict(dev.model_text)
positive_index = list(selected_model.classes_).index(1)
dev_probabilities = selected_model.predict_proba(dev.model_text.map(clean_text))[:, positive_index]
if not np.isfinite(dev_probabilities).all():
    raise RuntimeError("Non-finite development probabilities")
keyword_predictions, keyword_metrics = evaluate(y_dev, keyword_scores, 0.5)
dev_predictions, tfidf_metrics = evaluate(y_dev, dev_probabilities, selected_threshold)
metrics = {"keyword": keyword_metrics, "tfidf_logistic": tfidf_metrics}
print(json.dumps(metrics, indent=2))
predictions = dev[["complaint_id", "model_text", "escalation"]].copy()
predictions["keyword_score"] = keyword_scores
predictions["keyword_prediction"] = keyword_predictions
predictions["escalation_probability"] = dev_probabilities
predictions["escalation_prediction"] = dev_predictions
predictions["error_type"] = np.where((y_dev == 0) & (dev_predictions == 1), "false_positive",
                                      np.where((y_dev == 1) & (dev_predictions == 0), "false_negative", ""))
errors = predictions.loc[predictions.error_type.ne("")].copy()
display(errors)


## Versioned export

Only this cell writes files. Input hashes are rechecked. New directories are exclusively created; existing artifacts are never overwritten. The saved pipeline expects text normalized with `clean_text`; the exact ordered normalization rules are recorded in settings. No Python lambda or notebook function is embedded in the serialized estimator.

Load joblib artifacts only from trusted sources. Outputs retain source text for local analysis; review it before external sharing.


In [ ]:
for relative, expected in input_hashes.items():
    if sha256(ROOT / relative) != expected:
        raise RuntimeError(f"Input changed during run: {relative}")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
model_dir = ROOT / "models" / f"escalation_tfidf_{run_id}"
eval_dir = ROOT / "eval" / f"escalation_tfidf_{run_id}"
model_dir.mkdir(parents=True, exist_ok=False)
eval_dir.mkdir(parents=True, exist_ok=False)
joblib.dump(selected_model, model_dir / "pipeline.joblib")
metadata = {
    "target": "escalation", "target_definition": "likely escalation: explicit escalation or strong unresolved-service indicators",
    "label_mapping": {"0": "no_likely_escalation", "1": "likely_escalation"},
    "threshold": selected_threshold, "threshold_comparison": ">=", "selected_C": selected_c,
    "seed": SEED, "cv_splits": N_SPLITS, "cv_group": "connected author/post/normalized-text clusters",
    "cv_target": "escalation", "selection_metric": "pooled OOF positive-class F1",
    "tie_break": "smallest C, threshold nearest 0.50, then lower threshold",
    "C_candidates": C_VALUES, "threshold_candidates": THRESHOLDS,
    "tfidf": {"analyzer": "char_wb", "ngram_range": [3, 5]},
    "logistic_regression": {"class_weight": "balanced", "max_iter": 1000},
    "pipeline_input": "Apply ordered preprocessing below to model_text before predict_proba",
    "preprocessing": ["lowercase", "remove U+064B-U+065F and U+0670",
                      "replace أ إ آ ٱ with ا", "replace ة with ه", "replace ى with ي", "remove ـ"],
    "keyword_list": KEYWORDS, "keyword_score_type": "binary substring rule, not probability",
    "pr_auc_definition": "sklearn average_precision_score",
    "input_sha256": input_hashes, "fold_summary": fold_summary,
    "selected_oof_result": selected,
    "evaluation_split": "existing development validation; not an untouched test set",
    "versions": {"python": sys.version, "sklearn": sklearn.__version__,
                 "pandas": pd.__version__, "numpy": np.__version__, "joblib": joblib.__version__},
    "model_directory": str(model_dir), "evaluation_directory": str(eval_dir),
}
for path, contents in [(model_dir / "settings.json", metadata), (eval_dir / "metrics.json", metrics)]:
    with path.open("x", encoding="utf-8") as handle:
        json.dump(contents, handle, ensure_ascii=False, indent=2, allow_nan=False)
predictions.to_csv(eval_dir / "predictions.csv", index=False, mode="x")
errors.to_csv(eval_dir / "errors.csv", index=False, mode="x")
review_flags.to_csv(eval_dir / "training_label_review_flags.csv", index=False, mode="x")
pd.DataFrame(search_rows).to_csv(eval_dir / "cv_search.csv", index=False, mode="x")
oof_export = train[["complaint_id", "escalation", "author_group"]].copy()
oof_export["cv_group"] = groups
oof_export["selected_oof_probability"] = oof_by_c[selected_c]
oof_export["fold"] = 0
for fold, (_, idx) in enumerate(splits, 1):
    oof_export.loc[oof_export.index[idx], "fold"] = fold
oof_export.to_csv(eval_dir / "training_oof_predictions.csv", index=False, mode="x")
print("Model:", model_dir)
print("Evaluation:", eval_dir)


In [ ]:
import pandas as pd

rows = []

for name, result in metrics.items():
    rows.append({
        "Model": name,
        "Precision": result["positive_precision"],
        "Recall": result["positive_recall"],
        "F1": result["positive_f1"],
        "Macro-F1": result["macro_f1"],
        "PR-AUC": result["pr_auc_average_precision"]
    })

pd.DataFrame(rows)